# 🎲 Workshop: Bau deinen eigenen WM-Experten 2026
## Notebook 2: Simulation der WM 2026 mit Monte-Carlo

Warum können wir Fußballspiele nicht exakt vorhersagen? Weil der Sport voller Zufälle steckt! Ein Favorit kann durch eine rote Karte oder ein unglückliches Eigentor verlieren.

Um dennoch fundierte Prognosen zu treffen, nutzen Profis **Monte-Carlo-Simulationen**. Statt das Turnier nur einmal im Kopf durchzuspielen, lassen wir den Computer das Turnier **10.000-mal** simulieren. Aus der Häufigkeit der Ergebnisse berechnen wir die Wahrscheinlichkeiten (z.B. "In 15% aller Simulationen wird Frankreich Weltmeister").

### Lernziele:
1. Die ELO-Formel für Siegwahrscheinlichkeiten verstehen und in Python implementieren.
2. Ein einzelnes Fußballspiel simulieren.
3. Verstehen, warum mehr Simulationen (Gesetz der großen Zahlen) zu stabileren Ergebnissen führen.


### Schritt 1: Die ELO-Siegwahrscheinlichkeit
Die ELO-Zahl misst die Spielstärke eines Teams. Aus dem Unterschied zweier ELO-Zahlen ($R_A$ und $R_B$) lässt sich die mathematische Wahrscheinlichkeit berechnen, dass Team A gewinnt. Die Formel lautet:

$$P_A = \frac{1}{1 + 10^{\frac{R_B - R_A}{400}}}$$


In [ ]:
def win_probability(elo_a: float, elo_b: float) -> float:
    """Berechnet die Wahrscheinlichkeit, dass Team A gegen Team B gewinnt."""
    return 1.0 / (1.0 + 10 ** ((elo_b - elo_a) / 400))

# Beispiel: Deutschland (ELO 1900) gegen Ecuador (ELO 1750)
p_de = win_probability(1900, 1750)
print(f"Siegwahrscheinlichkeit Deutschland: {p_de * 100:.1f}%")
print(f"Siegwahrscheinlichkeit Ecuador: {(1 - p_de) * 100:.1f}%")


### Schritt 2: Ein einzelnes Spiel simulieren
Wir nutzen eine Zufallszahl zwischen `0.0` und `1.0` (`random.random()`). Liegt die Zufallszahl unter der Siegwahrscheinlichkeit von Team A, gewinnt Team A. Andernfalls gewinnt Team B.


In [ ]:
import random

def simulate_match(elo_a: float, elo_b: float) -> str:
    """Simuliert ein Spiel und gibt 'Team A' oder 'Team B' zurück."""
    p_a = win_probability(elo_a, elo_b)
    if random.random() < p_a:
        return "Team A"
    return "Team B"

# Simuliere das Spiel 5-mal. Jedes Mal kann ein anderes Ergebnis herauskommen!
for i in range(1, 6):
    print(f"Spiel {i} Sieger: {simulate_match(1900, 1750)}")


### Schritt 3: Gesetz der großen Zahlen (Varianz & Stabilität)
Wenn wir ein Spiel nur 10-mal simulieren, ist das Ergebnis sehr instabil. Simulieren wir es jedoch 10.000-mal, nähert sich das Ergebnis der exakten mathematischen Wahrscheinlichkeit an.
Wir probieren das interaktiv aus.


In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def run_simulation(elo_a, elo_b, n_sims):
    results = [simulate_match(elo_a, elo_b) for _ in range(n_sims)]
    wins_a = results.count("Team A")
    pct_a = (wins_a / n_sims) * 100
    
    print(f"Bei {n_sims} Simulationen:")
    print(f"  Team A gewinnt: {wins_a} Mal ({pct_a:.2f}%)")
    print(f"  Team B gewinnt: {n_sims - wins_a} Mal ({100 - pct_a:.2f}%)")
    print(f"  Ziel-Wahrscheinlichkeit: {win_probability(elo_a, elo_b)*100:.2f}%")

interact(run_simulation, 
         elo_a=widgets.IntSlider(min=1200, max=2200, step=50, value=1900, description="ELO A"),
         elo_b=widgets.IntSlider(min=1200, max=2200, step=50, value=1750, description="ELO B"),
         n_sims=widgets.IntSlider(min=10, max=10000, step=50, value=100, description="Sims"));


### Schritt 4: Turniersimulation auf dem Server
Unser MCP-Server simuliert nicht nur ein Spiel, sondern den gesamten WM-Spielplan inklusive Gruppenphase, Tordifferenz-Tiebreakern, K.o.-Runden und dem Finale.
Rufen wir das Tool erneut auf und testen, wie die Ergebnisse variieren.


In [ ]:
import httpx
MCP_URL = "https://wm2026.fh-swf.cloud/mcp"

def get_simulation(n_sims):
    response = httpx.post(f"{MCP_URL}/call", json={
        "tool": "simulate_tournament",
        "parameters": {"team": "Deutschland", "n_sims": n_sims}
    })
    return response.json().get("result", {})

print("=== 10 Simulationen ===")
print(get_simulation(10).get("summary"))
print("\n=== 5000 Simulationen ===")
print(get_simulation(5000).get("summary"))


### 🧠 Aufgabe für dich:
Welchen Einfluss hat der Heimvorteil? Simuliere das Turnier für die **USA** (Co-Gastgeber) einmal mit `home_boost=0` und einmal mit `home_boost=150` (jeweils 5000 Simulationen). Vergleiche die Titelchancen.


In [ ]:
# DEINE LÖSUNG HIER:
usa_no_boost = httpx.post(f"{MCP_URL}/call", json={
    "tool": "simulate_tournament",
    "parameters": {"team": "USA", "n_sims": 5000, "home_boost": 0}
}).json().get("result", {})

usa_with_boost = httpx.post(f"{MCP_URL}/call", json={
    "tool": "simulate_tournament",
    "parameters": {"team": "USA", "n_sims": 5000, "home_boost": 150}
}).json().get("result", {})

print("Ohne Heimvorteil:", usa_no_boost.get("summary"))
print("Mit Heimvorteil (Boost +150 ELO):", usa_with_boost.get("summary"))
